# WO-03 Q5 — nearest main stem
For a point: the nearest river reach whose upstream area (`upland_skm`, km²) is at or above a threshold, its distance, upstream area, Strahler order and mean discharge; compared with the containing L6 basin's upstream area (`basin06.up_area`). Thresholds 1,000, 10,000 and 100,000 km². Uses `gaz.rivers` (8,477,883 reaches); `gaz.hydrorivers` lacks 158,568 reaches in northern North America and Greenland, so it is not used. River names are Q4 (on hold). Read-only.

In [1]:
%matplotlib inline
# Cell 1 — Setup: thresholds, test places, the WO-02 sample points
import sys, time, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.whg import wo03_common as C

conn = db_connect()
conn.execute("SET statement_timeout = 120000")
THRESHOLDS = [1000, 10000, 100000]
SAMPLE = C.population_points()
n_reach = [conn.execute("SELECT count(*) FROM gaz.rivers WHERE upland_skm >= %s", (t,)).fetchone()[0] for t in THRESHOLDS]
print('commit', C.commit(), '| test places:', len(C.PLACES), '| sample points:', len(SAMPLE))
print('reaches at or above each threshold (km2):', dict(zip(THRESHOLDS, n_reach)))

commit 23d4144 | test places: 10 | sample points: 306
reaches at or above each threshold (km2): {1000: 905279, 10000: 273674, 100000: 60558}


In [2]:
# Cell 2 — Query functions, then the ten test places
STEM_SQL = """
WITH p AS (SELECT ST_SetSRID(ST_MakePoint(%s, %s), 4326) AS geom)
SELECT r.hyriv_id::bigint, r.upland_skm::float8, r.ord_stra::int, r.dis_av_cms::float8,
       ST_Distance(r.geog, p.geom::geography) / 1000.0 AS km
FROM gaz.rivers r, p
WHERE r.upland_skm >= %s
ORDER BY r.geog <-> p.geom::geography LIMIT 1
"""
BASIN_SQL = """
SELECT hybas_id::bigint, up_area::float8, sub_area::float8 FROM public.basin06
WHERE ST_Intersects(geom, ST_SetSRID(ST_MakePoint(%s, %s), 4326))
"""
INCORE_SQL = """
SELECT ST_Intersects(r.geom, b.geom) FROM gaz.rivers r, public.basin06 b WHERE r.hyriv_id = %s AND b.hybas_id = %s
"""
def stem_row(lat, lon, hid, up_area, thr):
    t0 = time.perf_counter()
    r = conn.execute(STEM_SQL, (lon, lat, thr)).fetchone()
    dt = time.perf_counter() - t0
    inside = bool(conn.execute(INCORE_SQL, (r[0], hid)).fetchone()[0]) if hid is not None else False
    return dict(threshold=thr, reach=r[0], km=r[4], stem_up_km2=r[1], order=r[2], disch_cms=r[3],
                ratio=(r[1] / up_area) if up_area else np.nan, in_core=inside, seconds=dt)

rows = []
for name, (lat, lon) in C.PLACES.items():
    b = conn.execute(BASIN_SQL, (lon, lat)).fetchone()
    hid, up, sub = (b if b else (None, np.nan, np.nan))
    for thr in THRESHOLDS:
        d = stem_row(lat, lon, hid, up, thr)
        d.update(place=name, core_id=hid, core_up_km2=up)
        rows.append(d)
T5 = pd.DataFrame(rows)
show = T5[['place', 'threshold', 'km', 'stem_up_km2', 'order', 'disch_cms', 'core_up_km2', 'ratio', 'in_core', 'seconds']].copy()
for c in ['km', 'stem_up_km2', 'disch_cms', 'core_up_km2']:
    show[c] = show[c].round(1)
show['ratio'] = show['ratio'].round(2)
show['seconds'] = show['seconds'].round(2)
print(show.to_string(index=False))

           place  threshold     km  stem_up_km2  order  disch_cms  core_up_km2  ratio  in_core  seconds
        Timbuktu       1000   12.6     378876.4      8      599.2     382644.3   0.99     True     0.02
        Timbuktu      10000   12.6     378876.4      8      599.2     382644.3   0.99     True     0.00
        Timbuktu     100000   12.6     378876.4      8      599.2     382644.3   0.99     True     0.00
         Tbilisi       1000    0.4      21515.5      6      172.9      23251.9   0.93     True     0.00
         Tbilisi      10000    0.4      21515.5      6      172.9      23251.9   0.93     True     0.00
         Tbilisi     100000  361.9     179564.1      7      684.4      23251.9   7.72    False     0.16
        Santa Fe       1000   25.5      36489.8      6       60.5      49334.1   0.74     True     0.00
        Santa Fe      10000   25.5      36489.8      6       60.5      49334.1   0.74     True     0.00
        Santa Fe     100000  447.4     115603.3      7      203.

In [3]:
# Cell 3 — The WO-02 sample (306 basin interior points) at the three thresholds; saved to a CSV
rows, t_start = [], time.perf_counter()
for k, s in enumerate(SAMPLE, 1):
    b = conn.execute("SELECT up_area::float8, sub_area::float8 FROM public.basin06 WHERE hybas_id = %s", (s['hybas_id_l6'],)).fetchone()
    for thr in THRESHOLDS:
        d = stem_row(s['lat'], s['lon'], s['hybas_id_l6'], b[0], thr)
        d.update(core_id=s['hybas_id_l6'], core_up_km2=b[0], core_km2=b[1], climate_zone=s['climate_zone'])
        rows.append(d)
    if k % 100 == 0:
        print(f'  {k} of {len(SAMPLE)} in {time.perf_counter() - t_start:.0f}s')
S5 = pd.DataFrame(rows)
S5.to_csv(C.OUT / 'q5_sample.csv', index=False)
print(f'rows saved: {len(S5)} | cores: {S5["core_id"].nunique()} | this run took {time.perf_counter() - t_start:.0f}s')
print('seconds per query:', {k: round(float(v), 3) for k, v in S5['seconds'].describe(percentiles=[.5, .9, .99]).items()})

rows saved: 918 | cores: 306 | this run took 33s
seconds per query: {'count': 918.0, 'mean': 0.031, 'std': 0.084, 'min': 0.0, '50%': 0.002, '90%': 0.088, '99%': 0.385, 'max': 1.242}


In [4]:
# Cell 4 — Sample distribution: distance, and the gap between the nearest main stem and the core's own upstream area
L = []
for thr in THRESHOLDS:
    d = S5[S5['threshold'] == thr]
    km = d['km'].to_numpy(); ra = d['ratio'].to_numpy()
    q = lambda a, p: float(np.percentile(a, p))
    L.append(f'threshold {thr} km2 (n={len(d)})')
    L.append(f'  distance to nearest such reach, km: 10% {q(km,10):.1f} | 50% {q(km,50):.1f} | 90% {q(km,90):.1f} | max {km.max():.1f}')
    L.append(f'  stem upstream area / core upstream area: 10% {q(ra,10):.2f} | 50% {q(ra,50):.2f} | 90% {q(ra,90):.1f} | max {ra.max():.0f}')
    L.append(f'  ratio at or above 10: {int((ra >= 10).sum())} ({100 * (ra >= 10).mean():.1f}%) | at or above 100: {int((ra >= 100).sum())} ({100 * (ra >= 100).mean():.1f}%) | at or above 1,000: {int((ra >= 1000).sum())}')
    L.append(f'  reach touches the core: {int(d["in_core"].sum())} ({100 * d["in_core"].mean():.1f}%); of those, ratio at or above 10: {int(((ra >= 10) & d["in_core"].to_numpy()).sum())}')
    L.append(f'  not touching the core, ratio at or above 10: {int(((ra >= 10) & ~d["in_core"].to_numpy()).sum())}; distance of those, km: median {np.median(km[(ra >= 10) & ~d["in_core"].to_numpy()]) if ((ra >= 10) & ~d["in_core"].to_numpy()).any() else float("nan"):.1f}')
    L.append('')
print('\n'.join(L))

threshold 1000 km2 (n=306)
  distance to nearest such reach, km: 10% 1.5 | 50% 9.0 | 90% 30.3 | max 618.7
  stem upstream area / core upstream area: 10% 0.05 | 50% 0.36 | 90% 1.6 | max 12184
  ratio at or above 10: 13 (4.2%) | at or above 100: 5 (1.6%) | at or above 1,000: 4
  reach touches the core: 234 (76.5%); of those, ratio at or above 10: 0
  not touching the core, ratio at or above 10: 13; distance of those, km: median 10.1

threshold 10000 km2 (n=306)
  distance to nearest such reach, km: 10% 4.8 | 50% 49.0 | 90% 155.1 | max 978.7
  stem upstream area / core upstream area: 10% 0.55 | 50% 1.47 | 90% 19.6 | max 68574
  ratio at or above 10: 46 (15.0%) | at or above 100: 12 (3.9%) | at or above 1,000: 4
  reach touches the core: 109 (35.6%); of those, ratio at or above 10: 0
  not touching the core, ratio at or above 10: 46; distance of those, km: median 61.9

threshold 100000 km2 (n=306)
  distance to nearest such reach, km: 10% 38.2 | 50% 317.5 | 90% 1005.8 | max 2628.4
  stem u